# Loading the Data

In [1]:
import os
import json

# Relative path from the notebook to the corpus directory
corpus_dir = '/home/jugal/Documents/DataScience/Advanced_RAG/vectordatabases/open_ragbench/pdf/arxiv/corpus'

# Get all JSON files in the directory and sort them alphabetically 
json_files = sorted([f for f in os.listdir(corpus_dir) if f.endswith('.json')])

# Get the first 100 files
first_100_files = json_files[:100]

loaded_data = []

# Load the content of each JSON file
for file_name in first_100_files:
    file_path = os.path.join(corpus_dir, file_name)
    with open(file_path, 'r', encoding='utf-8') as f:
        try:
            data = json.load(f)
            loaded_data.append(data)
        except json.JSONDecodeError as e:
            print(f"Error reading {file_name}: {e}")

print(f"Successfully loaded {len(loaded_data)} JSON files.")

# If you want to see the contents of the very first loaded file
# print(loaded_data[0])

import json

id_list = set()

for i in loaded_data:
    id_list.add(i["id"])

with open("/home/jugal/Documents/DataScience/Advanced_RAG/vectordatabases/open_ragbench/pdf/arxiv/qrels.json", "r") as qrels:
    qrels = json.load(qrels)

qrels_100 = {}
for key in qrels:
    if qrels[key]['doc_id'] in id_list:
        qrels_100[key] = qrels[key]

print("len(qrels_100) =", len(qrels_100))

with open("/home/jugal/Documents/DataScience/Advanced_RAG/vectordatabases/open_ragbench/pdf/arxiv/queries.json", "r") as queries:
    queries = json.load(queries)

queries_100 = {}

for qrel in qrels_100:
    queries_100[qrel] = queries[qrel]

print("len(queries_100) =", len(queries_100))


Successfully loaded 100 JSON files.
len(qrels_100) = 334
len(queries_100) = 334


# Loading models and Libraries

In [2]:
import json
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from tqdm.auto import tqdm
from collections import defaultdict
import numpy as np
from qdrant_client import models
from fastembed import SparseTextEmbedding

/home/jugal/miniconda3/envs/ds/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
client = QdrantClient(host="localhost", port=6333)

# Evaluation
### Quadrant Connection

In [33]:
key = 'dc064d11-cd18-4866-8a99-f16b0abec9c6'


query = queries_100[key]['query']

In [3]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("mixedbread-ai/mxbai-embed-large-v1", device="cpu")

I0000 00:00:1782583441.815589  178636 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1782583442.428935  178636 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1782583445.049488  178636 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1782583445.053630  178636 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.

In [ ]:
def evaluate_retrieval():
    k_values = [1, 2, 3, 4, 5]

    metrics = defaultdict(list)

    client = QdrantClient(host="localhost", port=6333)
    collection_name = "research_papers_v1"

    processed = 0

    for key in queries_100:

        query = queries_100[key]["query"]

        query_vector = embedding_model.encode(query)

        search_results = client.query_points(
            collection_name=collection_name,
            query=query_vector,
            limit=max(k_values),
            with_payload=True,
            with_vectors=False
        ).points

        retrieved_doc_ids = []

        for point in search_results:

            doc_id = point.payload.get("section_id") if point.payload else None

            if not doc_id:
                doc_id = point.id

            retrieved_doc_ids.append(str(doc_id))

        # Ground Truth
        ground_truth = qrels[key]
        ground_truth_doc_id = (
            str(ground_truth["section_id"]) +
            ground_truth["doc_id"]
        )

        # Recall@K
        for k in k_values:

            retrieved_k = retrieved_doc_ids[:k]

            hit = int(
                ground_truth_doc_id in retrieved_k
            )

            metrics[f"Recall@{k}"].append(hit)

        # Standard MRR
        rr = 0.0

        if ground_truth_doc_id in retrieved_doc_ids:
            rank = (
                retrieved_doc_ids.index(
                    ground_truth_doc_id
                ) + 1
            )
            rr = 1.0 / rank

        metrics["MRR"].append(rr)

        processed += 1

        if processed % 20 == 0:
            print(
                f"Processed {processed}/{len(queries_100)} queries..."
            )

    final_metrics = {
        metric: round(np.mean(values), 4)
        for metric, values in metrics.items()
    }

    return final_metrics

In [80]:
metrics = evaluate_retrieval()

Processed 20/334 queries...
Processed 40/334 queries...
Processed 60/334 queries...
Processed 80/334 queries...
Processed 100/334 queries...
Processed 120/334 queries...
Processed 140/334 queries...
Processed 160/334 queries...
Processed 180/334 queries...
Processed 200/334 queries...
Processed 220/334 queries...
Processed 240/334 queries...
Processed 260/334 queries...
Processed 280/334 queries...
Processed 300/334 queries...
Processed 320/334 queries...


# Using dence Vector Results

- 'Recall@1': np.float64(0.5958),
- 'Recall@2': np.float64(0.7246),
- 'Recall@3': np.float64(0.7904),
-  'Recall@4': np.float64(0.8443),
-  'Recall@5': np.float64(0.8653),
-  'MRR': np.float64(0.6998)}


###  MRR

An MRR of 0.6998 means that, on average, the correct document is being retrieved very close to the top of the ranking.

### Recall 

The sweet spot of recall is at Recall @4​


# Hybrid rag Evaluation

In [ ]:
def evaluate_hybrid_retrieval():
    k_values = [1, 2, 3, 4, 5]
    metrics = defaultdict(list)

    # 1. Initialize Clients
    # Load the sparse model (dense 'model' is assumed to be globally available like before)
    bm25_model = SparseTextEmbedding(model_name="Qdrant/bm25")
    
    # Use the local path instead of localhost server
    client = QdrantClient(path="./qdrant_vectorstore")
    collection_name = "research_papers_v2_hybrid"

    processed = 0
    # from sentence_transformers import SentenceTransformer

    # embedding_model = SentenceTransformer("mixedbread-ai/mxbai-embed-large-v1", device="cpu")

    for key in queries_100:
        query = queries_100[key]["query"]

        # 2. Generate Vectors
        dense_query_vector = embedding_model.encode(query)
        
        sparse_emb = list(bm25_model.embed([query]))[0]
        sparse_query_vector = models.SparseVector(
            indices=sparse_emb.indices.tolist(),
            values=sparse_emb.values.tolist()
        )

        # 3. Hybrid Search using RRF
        search_results = client.query_points(
            collection_name=collection_name,
            prefetch=[
                # Fetch Top 5 using Dense Embeddings
                models.Prefetch(
                    query=dense_query_vector,
                    using="dense",
                    limit=10,
                ),
                # Fetch Top 5 using Sparse (BM25) Embeddings
                models.Prefetch(
                    query=sparse_query_vector,
                    using="sparse",
                    limit=10,
                ),
            ],
            query=models.FusionQuery(fusion=models.Fusion.RRF),
            limit=10,
            with_payload=True,
            with_vectors=False
        ).points

        retrieved_doc_ids = []

        # Extract doc_ids as before
        for point in search_results:
            doc_id = point.payload.get("section_id") if point.payload else None
            if not doc_id:
                doc_id = point.id
            retrieved_doc_ids.append(str(doc_id))

        # Ground Truth Evaluation
        ground_truth = qrels[key]
        ground_truth_doc_id = (
            str(ground_truth["section_id"]) + ground_truth["doc_id"]
        )

        # Recall@K
        for k in k_values:
            retrieved_k = retrieved_doc_ids[:k]
            hit = int(ground_truth_doc_id in retrieved_k)
            metrics[f"Recall@{k}"].append(hit)

        # Standard MRR
        rr = 0.0
        if ground_truth_doc_id in retrieved_doc_ids:
            rank = retrieved_doc_ids.index(ground_truth_doc_id) + 1
            rr = 1.0 / rank

        metrics["MRR"].append(rr)

        processed += 1
        if processed % 20 == 0:
            print(f"Processed {processed}/{len(queries_100)} queries...")

    # Calculate final means
    final_metrics = {
        metric: round(np.mean(values), 4)
        for metric, values in metrics.items()
    }

    return final_metrics

# Run the evaluation
hybrid_metrics = evaluate_hybrid_retrieval()
print("Hybrid Search Results:", hybrid_metrics)


I0000 00:00:1782583360.649567  170309 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1782583361.125549  170309 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1782583363.446312  170309 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1782583363.449960  170309 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.

## Hybrid search Results(for Top 5)

Latency = 3 minutes 21 secs // 334 Queries

- 'Recall@1': np.float64(0.6766), 
- 'Recall@2': np.float64(0.8084), 
- 'Recall@3': np.float64(0.8772), 
- 'Recall@4': np.float64(0.9192), 
- 'Recall@5': np.float64(0.9371), 
- 'MRR': np.float64(0.7795)

## Hybrid search Results(for Top 10)

Latency = 3 minutes 21 secs // 334 Queries

- 'Recall@1': np.float64(0.6766), 
- 'Recall@2': np.float64(0.8084), 
- 'Recall@3': np.float64(0.8772), 
- 'Recall@4': np.float64(0.9192), 
- 'Recall@5': np.float64(0.9371), 
- 'MRR': np.float64(0.7795)

try:
    client.close()
except:
    pass


### Further Steps

- Cross Encoder
- Weighted fusion
- Relative Score Fusion
- Learned Sparse Embeddings (SPLADE)\
- Alpha Tunning
- Query Adaptive Alpha
- Retrieval Depth
- Reranker Models
- Late Interactiona and Colbert

# Cross Encoders

In [4]:
from FlagEmbedding import FlagReranker

reranker_model = FlagReranker(
    "BAAI/bge-reranker-v2-m3",
    use_fp16=False,
    devices="cpu"
)

In [5]:
print(reranker_model.target_devices)

['cpu']


Calculation:

for 50 chunks
- 334 queries
- 50 chunks retrived
- total pairs = 50 * 334 = 16700
- latency per pair = 3.1 sec
- Total Latency = 14 hours approx.

for 10 chunks
- 334 queries
- 10 chunks retrived
- total pairs = 10 * 334 = 3440
- latency per pair = 3.1 sec
- Total Latency = 2 hours approx.

In [27]:
10 * 20 * 3.1 

620.0

In [9]:
text = """Elon Reeve Musk (/ˈiːlɒn/ ⓘ EE-lon; born June 28, 1971) is a businessman and former public official who is the CEO and largest shareholder of Tesla and SpaceX. Musk has been the wealthiest person in the world since 2025, and became the first and only trillionaire in terms of US dollars in mid June 2026; however as of June 21, 2026, Forbes estimates his net worth to be US$951 billion.

Born into the wealthy Musk family in Pretoria, South Africa, Musk emigrated in 1989 to Canada; he has Canadian citizenship since his mother was born there. He received bachelor's degrees in 1997 from the University of Pennsylvania before moving to California to pursue business ventures. In 1995, Musk co-founded Zip2, a web software company. Following its sale in 1999, he co-founded X.com, an e-commerce payment system that merged with Confinity in March 2000 to form PayPal, which was acquired by eBay in 2002. Musk also became an American citizen in 2002.

In 2002, Musk founded and became CEO and chief engineer of SpaceX, a space technology company; the company has since led innovations in reusable rockets and commercial spaceflight. Musk joined Tesla as an early investor in 2004 and became its CEO and product architect in 2008; it has since become a leader in electric vehicles. In 2015, Musk co-founded OpenAI to advance artificial intelligence (AI) research, but later left; his growing discontent with the organization's direction and leadership in the AI boom in the 2020s led him to establish xAI, which became a subsidiary of SpaceX in 2026. In 2022, he acquired Twitter, a social networking service; he implemented significant changes and rebranded it as X in 2023. His other businesses include Neuralink, a neurotechnology company that he co-founded in 2016, and the Boring Company, a tunneling company that he founded in 2017. In November 2025, Tesla approved a pay package worth $1 trillion for Musk, which he is to receive over 10 years if certain milestones are met, such as achieving a market capitalization of $8.5 trillion.

Musk is a supporter of global far-right politics, figures, and political parties. He was the largest donor in the 2024 U.S. presidential election, where he supported Donald Trump. After Trump was inaugurated in January 2025, Musk served as Senior Advisor to the President and as the de facto head of the Department of Government Efficiency (DOGE). Musk left the Trump administration in May 2025 and returned to managing his companies; shortly thereafter he had a public feud with Trump.

Musk's political activities, statements and views have made him a polarizing figure. He has been criticized for making unscientific and misleading statements, including spreading COVID-19 misinformation, promoting conspiracy theories, and affirming antisemitic, white nationalist, racist, and transphobic comments. His acquisition of Twitter was controversial because, following his pledge to decrease censorship, there was an increase in hate speech and misinformation on the service. His role in the second Trump administration attracted public backlash, particularly in response to DOGE and its cuts to the US Agency for International Development (USAID).
"""
text = text[:1500]
query = "When does elon musk created SpaseX"

In [21]:
reranker_model.compute_score([text,query])

[2.1324703693389893]

In [26]:
def evaluate_hybrid_retrieval():
    k_values = [1, 2, 3, 4, 5]
    metrics = defaultdict(list)

    # 1. Initialize Clients
    # Load the sparse model (dense 'embedding_model' is assumed to be globally available like before)
    bm25_model = SparseTextEmbedding(model_name="Qdrant/bm25")
    
    # Use the local path instead of localhost server
    client = QdrantClient(host="localhost", port=6333)
    collection_name = "research_papers_v2_hybrid"

    processed = 0

    for key in queries_100:
        query = queries_100[key]["query"]

        # 2. Generate Vectors (Updated to use embedding_model)
        dense_query_vector = embedding_model.encode(query)
        
        sparse_emb = list(bm25_model.embed([query]))[0]
        sparse_query_vector = models.SparseVector(
            indices=sparse_emb.indices.tolist(),
            values=sparse_emb.values.tolist()
        )

        # 3. Hybrid Search using RRF (Take top 50)
        search_results = client.query_points(
            collection_name=collection_name,
            prefetch=[
                # Fetch Top 50 using Dense Embeddings
                models.Prefetch(
                    query=dense_query_vector,
                    using="dense",
                    limit=50,
                ),
                # Fetch Top 50 using Sparse (BM25) Embeddings
                models.Prefetch(
                    query=sparse_query_vector,
                    using="sparse",
                    limit=50,
                ),
            ],
            query=models.FusionQuery(fusion=models.Fusion.RRF),
            limit=10,
            with_payload=True,
            with_vectors=False
        ).points

        # 4. Rerank the top 50 results
                # 4. Rerank the top 50 results
        text_pairs = []
        for point in search_results:
            text = point.payload.get("text", "") if point.payload else ""
            text_pairs.append([query, text])

        # Compute reranker scores
        scores = reranker_model.compute_score(text_pairs)

        # Sort by reranker score (higher is better)
        reranked_results = sorted(
            zip(search_results, scores),
            key=lambda x: x[1],
            reverse=True
        )

        top_5_results = [result[0] for result in reranked_results[:max(k_values)]]
        retrieved_doc_ids = []

        # Extract doc_ids as before
        for point in top_5_results:
            doc_id = point.payload.get("section_id") if point.payload else None
            if not doc_id:
                doc_id = point.id
            retrieved_doc_ids.append(str(doc_id))

        # Ground Truth Evaluation
        ground_truth = qrels[key]
        ground_truth_doc_id = (
            str(ground_truth["section_id"]) + ground_truth["doc_id"]
        )

        # Recall@K
        for k in k_values:
            retrieved_k = retrieved_doc_ids[:k]
            hit = int(ground_truth_doc_id in retrieved_k)
            metrics[f"Recall@{k}"].append(hit)

        # Standard MRR
        rr = 0.0
        if ground_truth_doc_id in retrieved_doc_ids:
            rank = retrieved_doc_ids.index(ground_truth_doc_id) + 1
            rr = 1.0 / rank

        metrics["MRR"].append(rr)

        processed += 1
        if processed % 20 == 0:
            print(f"Processed {processed}/{len(queries_100)} queries...")

    # Calculate final means
    final_metrics = {
        metric: round(np.mean(values), 4)
        for metric, values in metrics.items()
    }

    return final_metrics

# Run the evaluation
hybrid_metrics = evaluate_hybrid_retrieval()
print("Hybrid Search Results:", hybrid_metrics)


Fetching 18 files: 100%|██████████| 18/18 [00:01<00:00, 12.45it/s]


Processed 20/334 queries...
Processed 40/334 queries...
Processed 60/334 queries...
Processed 80/334 queries...
Processed 100/334 queries...
Processed 120/334 queries...
Processed 140/334 queries...
Processed 160/334 queries...
Processed 180/334 queries...
Processed 200/334 queries...
Processed 220/334 queries...
Processed 240/334 queries...
Processed 260/334 queries...
Processed 280/334 queries...
Processed 300/334 queries...
Processed 320/334 queries...
Hybrid Search Results: {'Recall@1': np.float64(0.7455), 'Recall@2': np.float64(0.8653), 'Recall@3': np.float64(0.9012), 'Recall@4': np.float64(0.9281), 'Recall@5': np.float64(0.9491), 'MRR': np.float64(0.8283)}


In [28]:
hybrid_metrics

{'Recall@1': np.float64(0.7455),
 'Recall@2': np.float64(0.8653),
 'Recall@3': np.float64(0.9012),
 'Recall@4': np.float64(0.9281),
 'Recall@5': np.float64(0.9491),
 'MRR': np.float64(0.8283)}

# Putting the Results Together

## Using dence Vector Results

- 'Recall@1': np.float64(0.5958),
- 'Recall@2': np.float64(0.7246),
- 'Recall@3': np.float64(0.7904),
-  'Recall@4': np.float64(0.8443),
-  'Recall@5': np.float64(0.8653),
-  'MRR': np.float64(0.6998)

## Hybrid search Results

Latency = 3 minutes 21 secs // 334 Queries

- 'Recall@1': np.float64(0.6766), 
- 'Recall@2': np.float64(0.8084), 
- 'Recall@3': np.float64(0.8772), 
- 'Recall@4': np.float64(0.9192), 
- 'Recall@5': np.float64(0.9371), 
- 'MRR': np.float64(0.7795)


## Cross Encoder

Latency = 161 min for 10 chunks retrival per query

- 'Recall@1': np.float64(0.7455),
- 'Recall@2': np.float64(0.8653),
- 'Recall@3': np.float64(0.9012),
- 'Recall@4': np.float64(0.9281),
- 'Recall@5': np.float64(0.9491),
- 'MRR': np.float64(0.8283)

# Colbert lateinteraction

In [5]:
from qdrant_client import QdrantClient
from qdrant_client import models
from fastembed import LateInteractionTextEmbedding, SparseTextEmbedding
from collections import defaultdict
import numpy as np

def evaluate_colbert_hybrid_retrieval():
    k_values = [1, 2, 3, 4, 5]
    metrics = defaultdict(list)

    # 1. Initialize Clients and Models
    print("Initializing models...")
    colbert_model = LateInteractionTextEmbedding(model_name="colbert-ir/colbertv2.0")
    bm25_model = SparseTextEmbedding(model_name="Qdrant/bm25")
    
    client = QdrantClient(host="localhost", port=6333)
    collection_name = "research_papers_v3_Colbert" # Use the newly created collection

    processed = 0

    for key in queries_100:
        query = queries_100[key]["query"]

        # 2. Generate Vectors
        # Important: Use query_embed() for LateInteraction queries
        colbert_emb = list(colbert_model.query_embed(query))[0]
        # Ensure it's a python list of lists for Qdrant
        colbert_query_vector = colbert_emb.tolist() if hasattr(colbert_emb, 'tolist') else colbert_emb
        
        # Sparse (BM25) embedding
        sparse_emb = list(bm25_model.query_embed(query))[0] # query_embed also recommended for sparse
        sparse_query_vector = models.SparseVector(
            indices=sparse_emb.indices.tolist() if hasattr(sparse_emb.indices, 'tolist') else sparse_emb.indices,
            values=sparse_emb.values.tolist() if hasattr(sparse_emb.values, 'tolist') else sparse_emb.values,
        )

        # 3. Hybrid Search using RRF (No Cross-Encoder needed!)
        search_results = client.query_points(
            collection_name=collection_name,
            prefetch=[
                # Fetch Top 10 using ColBERT
                models.Prefetch(
                    query=colbert_query_vector,
                    using="colbert",
                    limit=10,
                ),
                # Fetch Top 10 using Sparse (BM25) Embeddings
                models.Prefetch(
                    query=sparse_query_vector,
                    using="bm25",
                    limit=10,
                ),
            ],
            query=models.FusionQuery(fusion=models.Fusion.RRF),
            limit=max(k_values), # Directly limit to Top 5 since ColBERT handles the deep interaction
            with_payload=True,
            with_vectors=False
        ).points

        # 4. Extract doc_ids directly from the RRF results
        retrieved_doc_ids = []
        for point in search_results:
            doc_id = point.payload.get("section_id") if point.payload else None
            if not doc_id:
                doc_id = point.id
            retrieved_doc_ids.append(str(doc_id))

        # Ground Truth Evaluation
        ground_truth = qrels[key]
        ground_truth_doc_id = (
            str(ground_truth["section_id"]) + ground_truth["doc_id"]
        )

        # Recall@K
        for k in k_values:
            retrieved_k = retrieved_doc_ids[:k]
            hit = int(ground_truth_doc_id in retrieved_k)
            metrics[f"Recall@{k}"].append(hit)

        # Standard MRR
        rr = 0.0
        if ground_truth_doc_id in retrieved_doc_ids:
            rank = retrieved_doc_ids.index(ground_truth_doc_id) + 1
            rr = 1.0 / rank

        metrics["MRR"].append(rr)

        processed += 1
        if processed % 20 == 0:
            print(f"Processed {processed}/{len(queries_100)} queries...")

    # Calculate final means
    final_metrics = {
        metric: round(np.mean(values), 4)
        for metric, values in metrics.items()
    }

    return final_metrics

# Run the evaluation
colbert_hybrid_metrics = evaluate_colbert_hybrid_retrieval()
print("ColBERT Hybrid Search Results:", colbert_hybrid_metrics)


Initializing models...
Processed 20/334 queries...
Processed 40/334 queries...
Processed 60/334 queries...
Processed 80/334 queries...
Processed 100/334 queries...
Processed 120/334 queries...
Processed 140/334 queries...
Processed 160/334 queries...
Processed 180/334 queries...
Processed 200/334 queries...
Processed 220/334 queries...
Processed 240/334 queries...
Processed 260/334 queries...
Processed 280/334 queries...
Processed 300/334 queries...
Processed 320/334 queries...
ColBERT Hybrid Search Results: {'Recall@1': np.float64(0.7096), 'Recall@2': np.float64(0.8473), 'Recall@3': np.float64(0.8862), 'Recall@4': np.float64(0.9222), 'Recall@5': np.float64(0.9311), 'MRR': np.float64(0.8022)}


In [7]:
colbert_hybrid_metrics

{'Recall@1': np.float64(0.7096),
 'Recall@2': np.float64(0.8473),
 'Recall@3': np.float64(0.8862),
 'Recall@4': np.float64(0.9222),
 'Recall@5': np.float64(0.9311),
 'MRR': np.float64(0.8022)}

## Using dence Vector Results

- 'Recall@1': np.float64(0.5958),
- 'Recall@2': np.float64(0.7246),
- 'Recall@3': np.float64(0.7904),
-  'Recall@4': np.float64(0.8443),
-  'Recall@5': np.float64(0.8653),
-  'MRR': np.float64(0.6998)

## Hybrid search Results

Latency = 3 minutes 21 secs // 334 Queries

- 'Recall@1': np.float64(0.6766), 
- 'Recall@2': np.float64(0.8084), 
- 'Recall@3': np.float64(0.8772), 
- 'Recall@4': np.float64(0.9192), 
- 'Recall@5': np.float64(0.9371), 
- 'MRR': np.float64(0.7795)


## Cross Encoder

Latency = 161 min for 10 chunks retrival per query

- 'Recall@1': np.float64(0.7455),
- 'Recall@2': np.float64(0.8653),
- 'Recall@3': np.float64(0.9012),
- 'Recall@4': np.float64(0.9281),
- 'Recall@5': np.float64(0.9491),
- 'MRR': np.float64(0.8283)


## Colbert
Latency = 1 min overall
- 'Recall@1': np.float64(0.7096),
- 'Recall@2': np.float64(0.8473),
- 'Recall@3': np.float64(0.8862),
- 'Recall@4': np.float64(0.9222),
- 'Recall@5': np.float64(0.9311),
- 'MRR': np.float64(0.8022)

## Observation

- Cross Encoder is only worth It, For Top 1 chunk 
- Colbert is a good choise for top 2 chunks
- For 3, 4 and 5 chunks Hybrid Search results are giving enough results.
- Only Dense Retrival should not be used
- If we find a way to getting the Good chunk to go above then the Hybrid search results can be as good as Cross Encoder and Colbert with low cost and latency
